# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Name:** Aryan Kadian
**Roll Number:** LCS2023051
**Dataset:** `sales.csv`
**Environment:** Google Colab + PySpark

This notebook builds a simple two-layer Data Lake:

```
data_lake/
|-- bronze/
|   `-- sales.csv              <- raw data, kept exactly as provided
`-- silver/
    |-- preprocessed_sales/    <- Spark output (CSV part file + _SUCCESS)
    `-- sales_silver.csv       <- the same Silver data as a single CSV file
```

The raw file is kept unchanged in **Bronze**, basic preprocessing is done with **Apache Spark**, and the result is saved in **Silver**. Every discrepancy count shown below is computed with Spark operations, not hard-coded.

## 1. Setup: install PySpark and create the Data Lake folders

In [1]:
# PySpark is usually preinstalled on Colab. It is installed only if missing.
try:
    import pyspark
except ImportError:
    !pip install -q pyspark
    import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.9


In [2]:
import os
import shutil
import hashlib
import urllib.request

LAKE_DIR   = "data_lake"
BRONZE_DIR = os.path.join(LAKE_DIR, "bronze")
SILVER_DIR = os.path.join(LAKE_DIR, "silver")
SILVER_OUT = os.path.join(SILVER_DIR, "preprocessed_sales")
BRONZE_FILE = os.path.join(BRONZE_DIR, "sales.csv")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

# Get the raw dataset into the Bronze layer. The file is copied or downloaded
# byte for byte, never opened and rewritten, so Bronze stays exactly as provided.
DATASET_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
               "data-mining-data-lake-assignment/main/dataset/sales.csv")
LOCAL_CANDIDATES = ["sales.csv", "dataset/sales.csv", "../../dataset/sales.csv"]

if not os.path.exists(BRONZE_FILE):
    local = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)
    if local:
        shutil.copyfile(local, BRONZE_FILE)
        print(f"Copied {local} -> {BRONZE_FILE}")
    else:
        urllib.request.urlretrieve(DATASET_URL, BRONZE_FILE)
        print(f"Downloaded {DATASET_URL} -> {BRONZE_FILE}")

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

BRONZE_MD5 = md5(BRONZE_FILE)
print("Bronze file:", BRONZE_FILE, "| size:", os.path.getsize(BRONZE_FILE), "bytes | md5:", BRONZE_MD5)

Downloaded https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv -> data_lake/bronze/sales.csv
Bronze file: data_lake/bronze/sales.csv | size: 109537 bytes | md5: 2b7d6737635600be42aa639140cd6ea9


In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = (SparkSession.builder
         .appName("SalesDataLake")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.9


## 2. Load and inspect the Bronze data with Spark

Every column is read as a **string** (`inferSchema=False`). Values like `not-a-date` or badly formatted numbers are kept as they are, so we can find them first and convert types afterwards.

In [4]:
bronze_df = (spark.read
             .option("header", True)
             .option("inferSchema", False)
             .csv(BRONZE_FILE))

print("Schema (raw, all strings):")
bronze_df.printSchema()

Schema (raw, all strings):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
print("First rows:")
bronze_df.show(10, truncate=False)

raw_count = bronze_df.count()
print("Total records in Bronze:", raw_count)
print("Total columns:", len(bronze_df.columns))

First rows:


+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

Total records in Bronze: 1000
Total columns: 13


In [6]:
# Quick look at what the numeric columns contain when cast to numbers.
# try_cast returns NULL instead of failing on a value it cannot convert.
(bronze_df
 .select(*[F.expr(f"try_cast({c} as double)").alias(c)
           for c in ["quantity", "unit_price", "discount_percent"]])
 .describe()
 .show())

+-------+------------------+------------------+------------------+
|summary|          quantity|        unit_price|  discount_percent|
+-------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|
|   mean|             5.355|            5702.4|            19.948|
| stddev|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|              -2.0|            -500.0|              -5.0|
|    max|              10.0|           81010.0|             120.0|
+-------+------------------+------------------+------------------+



## 3. Identify data discrepancies (all counts computed by Spark)

### 3.1 Duplicates

In [7]:
distinct_count = bronze_df.distinct().count()
exact_dupes = raw_count - distinct_count
print(f"Exact duplicate rows (extra copies): {exact_dupes}")

print("Rows that appear more than once:")
(bronze_df.groupBy(bronze_df.columns).count()
 .filter("count > 1")
 .select("order_id", "customer_id", "product", "order_date", "count")
 .orderBy("order_id")
 .show(truncate=False))

Exact duplicate rows (extra copies): 10
Rows that appear more than once:


+--------+-----------+-----------------+----------+-----+
|order_id|customer_id|product          |order_date|count|
+--------+-----------+-----------------+----------+-----+
|100055  |C0270      |Dry Fruits Pack  |2026-08-20|2    |
|100145  |C0007      |Face Wash        |2025-08-28|2    |
|100297  |C0149      |Dry Fruits Pack  |2025-07-29|2    |
|100317  |C0264      |Kurta            |2025-02-04|2    |
|100329  |C0151      |Bedside Table    |2025-04-08|2    |
|100444  |C0090      |Novel            |2025-09-11|2    |
|100502  |C0068      |Induction Cooktop|2025-03-11|2    |
|100588  |C0228      |Bookshelf        |2026-08-09|2    |
|100692  |C0141      |Electric Kettle  |2026-08-18|2    |
|100773  |C0315      |Study Table      |2026-02-25|2    |
+--------+-----------+-----------------+----------+-----+



In [8]:
# After exact duplicates are removed, some order_ids are still repeated
# but with different content. These are different orders, not copies.
dedup_tmp = bronze_df.dropDuplicates()
repeated_ids = (dedup_tmp.groupBy("order_id").count().filter("count > 1"))
print("order_ids shared by different (non-identical) rows:", repeated_ids.count())
(dedup_tmp.join(repeated_ids.select("order_id"), "order_id")
 .select("order_id", "customer_id", "customer_name", "product", "order_date")
 .orderBy("order_id")
 .show(truncate=False))

order_ids shared by different (non-identical) rows: 8


+--------+-----------+------------------+----------------------+----------+
|order_id|customer_id|customer_name     |product               |order_date|
+--------+-----------+------------------+----------------------+----------+
|100064  |C0149      |Manish Pillai     |Basmati Rice 5kg      |2026-08-31|
|100064  |C0276      |Ananya Agarwal    |Microwave Oven        |2025-05-15|
|100072  |C0312      |Aarav Kulkarni    |Face Wash             |2026-08-15|
|100072  |C0211      |Meera Chopra      |Competitive Exam Guide|2026-06-30|
|100305  |C0076      |Neha Reddy        |Office Chair          |2025-12-29|
|100305  |C0199      |Pooja Reddy       |Cooking Oil 5L        |2025-06-16|
|100331  |C0137      |Riya Kapoor       |Jacket                |2025-02-20|
|100331  |C0005      |Sakshi Joshi      |Football              |2025-06-06|
|100434  |C0149      |Manish Pillai     |Sugar 5kg             |2026-06-06|
|100434  |C0313      |Sai Das           |Business Book         |2026-03-30|
|100474  |C0

### 3.2 Null / empty values

In [9]:
# Spark reads empty CSV fields as NULL. Blank strings are counted as well, just in case.
null_counts = bronze_df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
])
null_counts.show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



### 3.3 Text-format inconsistencies (extra spaces / capitalization)

In [10]:
TEXT_COLS = ["customer_name", "product", "category", "payment_method",
             "city", "state", "order_status"]
# Columns whose correct form is Title Case. product and payment_method are left
# out because they contain acronyms such as "USB-C" and "UPI".
TITLE_CASE_COLS = ["customer_name", "category", "city", "state", "order_status"]

def normalise_spaces(col):
    return F.trim(F.regexp_replace(col, r"\s+", " "))

rows = []
for c in TEXT_COLS:
    col = F.col(c)
    space_issue = F.when(col.isNotNull() & (col != normalise_spaces(col)), 1).otherwise(0)
    if c in TITLE_CASE_COLS:
        case_issue = F.when(col.isNotNull() & (normalise_spaces(col) != F.initcap(normalise_spaces(col))), 1).otherwise(0)
    else:
        case_issue = F.lit(0)
    rows.append(bronze_df.select(F.lit(c).alias("column"),
                                 F.sum(space_issue).alias("extra_spaces"),
                                 F.sum(case_issue).alias("bad_capitalization")))

text_issues = rows[0]
for r in rows[1:]:
    text_issues = text_issues.unionByName(r)
text_issues.show(truncate=False)

+--------------+------------+------------------+
|column        |extra_spaces|bad_capitalization|
+--------------+------------+------------------+
|customer_name |3           |0                 |
|product       |0           |0                 |
|category      |0           |6                 |
|payment_method|0           |0                 |
|city          |3           |6                 |
|state         |0           |0                 |
|order_status  |0           |0                 |
+--------------+------------+------------------+



In [11]:
# Example: the raw category and city values, showing the inconsistent spellings.
for c in ["category", "city"]:
    print(f"Distinct raw values of '{c}' (quotes show extra spaces):")
    (bronze_df.groupBy(F.concat(F.lit("'"), F.col(c), F.lit("'")).alias(c))
     .count().orderBy(c).show(50, truncate=False))

Distinct raw values of 'category' (quotes show extra spaces):


+-----------------+-----+
|category         |count|
+-----------------+-----+
|'Beauty'         |136  |
|'Books'          |128  |
|'CLOTHING'       |1    |
|'Clothing'       |108  |
|'ELECTRONICS'    |1    |
|'Electronics'    |103  |
|'Furniture'      |132  |
|'Grocery'        |136  |
|'Home Appliances'|115  |
|'SPORTS'         |1    |
|'Sports'         |136  |
|'beauty'         |2    |
|'sports'         |1    |
+-----------------+-----+

Distinct raw values of 'city' (quotes show extra spaces):
+-------------+-----+
|city         |count|
+-------------+-----+
|NULL         |8    |
|' Hyderabad '|2    |
|' Mumbai '   |1    |
|'Bengaluru'  |91   |
|'Chennai'    |88   |
|'Coimbatore' |81   |
|'DELHI'      |2    |
|'Delhi'      |81   |
|'HYDERABAD'  |1    |
|'Hyderabad'  |94   |
|'JAIPUR'     |1    |
|'Jaipur'     |84   |
|'Kolkata'    |81   |
|'Lucknow'    |90   |
|'Mumbai'     |106  |
|'Pune'       |82   |
|'Vijayawada' |105  |
|'coimbatore' |1    |
|'delhi'      |1    |
+-------------+

### 3.4 Invalid numerical values

Validation rules used:
* `quantity` must be a whole number **> 0**
* `unit_price` must be a number **> 0**
* `discount_percent` must be between **0 and 100**

In [12]:
q  = F.expr("try_cast(quantity as double)")
up = F.expr("try_cast(unit_price as double)")
dp = F.expr("try_cast(discount_percent as double)")

numeric_issues = bronze_df.select(
    F.sum(F.when(F.col("quantity").isNotNull() & q.isNull(), 1).otherwise(0)).alias("quantity_not_numeric"),
    F.sum(F.when(q <= 0, 1).otherwise(0)).alias("quantity_<=_0"),
    F.sum(F.when(q.isNotNull() & (q != F.floor(q)), 1).otherwise(0)).alias("quantity_not_whole"),
    F.sum(F.when(F.col("unit_price").isNotNull() & up.isNull(), 1).otherwise(0)).alias("price_not_numeric"),
    F.sum(F.when(up <= 0, 1).otherwise(0)).alias("unit_price_<=_0"),
    F.sum(F.when(F.col("discount_percent").isNotNull() & dp.isNull(), 1).otherwise(0)).alias("discount_not_numeric"),
    F.sum(F.when((dp < 0) | (dp > 100), 1).otherwise(0)).alias("discount_out_of_0_100"),
)
numeric_issues.show(truncate=False)

print("Rows with at least one invalid numeric value:")
bronze_df.filter((q <= 0) | (up <= 0) | (dp < 0) | (dp > 100)) \
    .select("order_id", "product", "quantity", "unit_price", "discount_percent") \
    .show(50, truncate=False)

+--------------------+-------------+------------------+-----------------+---------------+--------------------+---------------------+
|quantity_not_numeric|quantity_<=_0|quantity_not_whole|price_not_numeric|unit_price_<=_0|discount_not_numeric|discount_out_of_0_100|
+--------------------+-------------+------------------+-----------------+---------------+--------------------+---------------------+
|0                   |12           |0                 |0                |8              |0                   |8                    |
+--------------------+-------------+------------------+-----------------+---------------+--------------------+---------------------+

Rows with at least one invalid numeric value:
+--------+----------------------+--------+----------+----------------+
|order_id|product               |quantity|unit_price|discount_percent|
+--------+----------------------+--------+----------+----------------+
|100034  |Yoga Mat              |7       |1720      |-5              |
|100

### 3.5 Inconsistent / invalid dates

The expected format is `yyyy-MM-dd`. Each raw value is sorted into a format pattern by regex. Then the value is parsed with `try_to_timestamp`, which returns NULL for dates that do not exist on the calendar (e.g. month 13 or 31 February).

In [13]:
def safe_date(fmt):
    # Parse order_date with the given pattern. Returns NULL (no error) when the value
    # is not a real calendar date. Works on Spark 3.4+ and 4.x.
    return F.expr(f"to_date(try_to_timestamp(order_date, '{fmt}'))")

date_pattern = (F.when(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
                 .when(F.col("order_date").rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
                 .when(F.col("order_date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
                 .when(F.col("order_date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "MM-dd-yyyy")
                 .when(F.col("order_date").isNull(), "NULL")
                 .otherwise("unrecognised"))

print("Date formats found in order_date:")
bronze_df.groupBy(date_pattern.alias("date_format")).count().orderBy(F.desc("count")).show(truncate=False)

print("Values that are not in the standard yyyy-MM-dd format, or are not real calendar dates:")
(bronze_df
 .filter((date_pattern != "yyyy-MM-dd") |
         safe_date("yyyy-MM-dd").isNull())
 .select("order_id", "order_date", date_pattern.alias("detected_format"))
 .show(truncate=False))

Date formats found in order_date:


+------------+-----+
|date_format |count|
+------------+-----+
|yyyy-MM-dd  |996  |
|yyyy/MM/dd  |1    |
|unrecognised|1    |
|MM-dd-yyyy  |1    |
|dd/MM/yyyy  |1    |
+------------+-----+

Values that are not in the standard yyyy-MM-dd format, or are not real calendar dates:
+--------+----------+---------------+
|order_id|order_date|detected_format|
+--------+----------+---------------+
|100097  |31/02/2026|dd/MM/yyyy     |
|100358  |2025/07/14|yyyy/MM/dd     |
|100448  |not-a-date|unrecognised   |
|100634  |2025-02-30|yyyy-MM-dd     |
|100637  |2026-13-05|yyyy-MM-dd     |
|100969  |09-15-2026|MM-dd-yyyy     |
+--------+----------+---------------+



### 3.6 Summary of discrepancies found

In [14]:
nulls_row = null_counts.first().asDict()
num_row = numeric_issues.first().asDict()
bad_dates = bronze_df.filter(
    (date_pattern != "yyyy-MM-dd") | safe_date("yyyy-MM-dd").isNull()
).count()
text_row = text_issues.agg(F.sum("extra_spaces").alias("s"), F.sum("bad_capitalization").alias("c")).first()

summary = [
    ("Exact duplicate rows", exact_dupes),
    ("Repeated order_id with different content", repeated_ids.count()),
    ("Missing customer_name", nulls_row["customer_name"]),
    ("Missing payment_method", nulls_row["payment_method"]),
    ("Missing city", nulls_row["city"]),
    ("Text values with extra spaces", text_row["s"]),
    ("Text values with inconsistent capitalization", text_row["c"]),
    ("quantity <= 0", num_row["quantity_<=_0"]),
    ("unit_price <= 0", num_row["unit_price_<=_0"]),
    ("discount_percent outside 0-100", num_row["discount_out_of_0_100"]),
    ("Non-standard or invalid order_date", bad_dates),
]
spark.createDataFrame(summary, ["issue", "count"]).show(truncate=False)

+--------------------------------------------+-----+
|issue                                       |count|
+--------------------------------------------+-----+
|Exact duplicate rows                        |10   |
|Repeated order_id with different content    |8    |
|Missing customer_name                       |10   |
|Missing payment_method                      |8    |
|Missing city                                |8    |
|Text values with extra spaces               |6    |
|Text values with inconsistent capitalization|12   |
|quantity <= 0                               |12   |
|unit_price <= 0                             |8    |
|discount_percent outside 0-100              |8    |
|Non-standard or invalid order_date          |6    |
+--------------------------------------------+-----+



## 4. Basic preprocessing → Silver layer

The steps follow the assignment order. The aim is basic, reasonable cleaning that **keeps as many useful records as possible**. It does not try to make the data perfect.

### Step 1 — Remove exact duplicate rows

In [15]:
before_count = bronze_df.count()
step1_df = bronze_df.dropDuplicates()
print(f"Rows before: {before_count} | after removing exact duplicates: {step1_df.count()}")

Rows before: 1000 | after removing exact duplicates: 990


### Step 3 (done before Step 2) — Trim spaces and standardize capitalization

Text is cleaned **before** missing values are filled. This way `' Mumbai '`, `'MUMBAI'` and `'Mumbai'` are treated as the same value, and blank strings become proper NULLs.

In [16]:
step3_df = step1_df
for c in TEXT_COLS:
    cleaned = normalise_spaces(F.col(c))
    if c in TITLE_CASE_COLS:
        cleaned = F.initcap(cleaned)
    # an empty string after trimming counts as missing
    step3_df = step3_df.withColumn(c, F.when(cleaned == "", None).otherwise(cleaned))

# ID columns: just trim them
for c in ["order_id", "customer_id"]:
    step3_df = step3_df.withColumn(c, F.trim(F.col(c)))

print("Category values after standardization:")
step3_df.groupBy("category").count().orderBy("category").show(truncate=False)
print("City values after standardization:")
step3_df.groupBy("city").count().orderBy("city").show(truncate=False)

Category values after standardization:


+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+

City values after standardization:


+----------+-----+
|city      |count|
+----------+-----+
|NULL      |8    |
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |83   |
|Hyderabad |97   |
|Jaipur    |83   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Vijayawada|104  |
+----------+-----+



### Step 2 — Handle missing values

* **customer_name**: the same `customer_id` usually appears in other orders with a name. The missing name is filled from those rows. If no name is known for that customer, it is set to `"Unknown"`.
* **payment_method** and **city**: these cannot be worked out reliably (for example, one state has several cities), so they are filled with the label `"Unknown"`. The record is kept.

In [17]:
known_names = (step3_df.filter(F.col("customer_name").isNotNull())
               .groupBy("customer_id")
               .agg(F.first("customer_name", ignorenulls=True).alias("known_name")))

names_recovered = (step3_df.filter(F.col("customer_name").isNull())
                   .join(known_names, "customer_id").count())
print("Missing customer names that can be recovered from the same customer_id:", names_recovered)

step2_df = (step3_df.join(known_names, "customer_id", "left")
            .withColumn("customer_name", F.coalesce("customer_name", "known_name", F.lit("Unknown")))
            .drop("known_name")
            .fillna({"payment_method": "Unknown", "city": "Unknown"})
            .select(bronze_df.columns))          # keep the original column order

step2_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                 for c in ["customer_name", "payment_method", "city"]]).show()

Missing customer names that can be recovered from the same customer_id: 10


+-------------+--------------+----+
|customer_name|payment_method|city|
+-------------+--------------+----+
|            0|             0|   0|
+-------------+--------------+----+



### Step 4 — Validate quantity, unit price and discount percentage

The columns are converted to numeric types. A clearly invalid value is **set to NULL instead of deleting the row**, because the rest of the record (customer, product, date, status) is still useful:

* `quantity` → integer; values `<= 0` become NULL
* `unit_price` → double; values `<= 0` become NULL
* `discount_percent` → double; values outside `0–100` become NULL

In [18]:
qty = F.expr("try_cast(quantity as double)")
price = F.expr("try_cast(unit_price as double)")
disc = F.expr("try_cast(discount_percent as double)")

step4_df = (step2_df
    .withColumn("quantity", F.when((qty > 0) & (qty == F.floor(qty)), qty.cast("int")))
    .withColumn("unit_price", F.when(price > 0, price))
    .withColumn("discount_percent", F.when((disc >= 0) & (disc <= 100), disc)))

step4_df.select(
    F.sum(F.col("quantity").isNull().cast("int")).alias("quantity_set_null"),
    F.sum(F.col("unit_price").isNull().cast("int")).alias("unit_price_set_null"),
    F.sum(F.col("discount_percent").isNull().cast("int")).alias("discount_set_null"),
).show()
step4_df.select("quantity", "unit_price", "discount_percent").summary("min", "max").show()

+-----------------+-------------------+-----------------+
|quantity_set_null|unit_price_set_null|discount_set_null|
+-----------------+-------------------+-----------------+
|               12|                  8|                8|
+-----------------+-------------------+-----------------+

+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|    min|       1|     190.0|             0.0|
|    max|      10|   81010.0|            40.0|
+-------+--------+----------+----------------+



### Step 5 — Convert dates to one format and handle invalid dates

Each recognised format is parsed with its own pattern:

* `yyyy-MM-dd` and `yyyy/MM/dd` (year first, no ambiguity)
* `dd/MM/yyyy` (day/month/year with slashes, the usual Indian format)
* `MM-dd-yyyy` (month-day-year with dashes)

All valid dates are stored as a Spark `DateType`, which is written as `yyyy-MM-dd`. Values that cannot be a real date (`not-a-date`, month 13, 31 February) become NULL. The row itself is kept.

In [19]:
raw_date = F.col("order_date")
parsed_date = (F.when(raw_date.rlike(r"^\d{4}-\d{2}-\d{2}$"), safe_date("yyyy-MM-dd"))
                .when(raw_date.rlike(r"^\d{4}/\d{2}/\d{2}$"), safe_date("yyyy/MM/dd"))
                .when(raw_date.rlike(r"^\d{2}/\d{2}/\d{4}$"), safe_date("dd/MM/yyyy"))
                .when(raw_date.rlike(r"^\d{2}-\d{2}-\d{4}$"), safe_date("MM-dd-yyyy")))

step5_df = step4_df.withColumn("order_date_clean", parsed_date)

print("How the non-standard dates were handled:")
(step5_df.filter(~raw_date.rlike(r"^\d{4}-\d{2}-\d{2}$") | F.col("order_date_clean").isNull())
 .select("order_id", F.col("order_date").alias("raw_order_date"), "order_date_clean")
 .show(truncate=False))

step5_df = step5_df.withColumn("order_date", F.col("order_date_clean")).drop("order_date_clean")

How the non-standard dates were handled:


+--------+--------------+----------------+
|order_id|raw_order_date|order_date_clean|
+--------+--------------+----------------+
|100448  |not-a-date    |NULL            |
|100358  |2025/07/14    |2025-07-14      |
|100097  |31/02/2026    |NULL            |
|100637  |2026-13-05    |NULL            |
|100969  |09-15-2026    |2026-09-15      |
|100634  |2025-02-30    |NULL            |
+--------+--------------+----------------+



### Step 6 — Final check, then keep useful records

* A second `dropDuplicates()` is run. After trimming and changing capitalization, two rows that differed only in formatting would now be identical copies.
* No other rows are removed. Rows with a repeated `order_id` but different content are **kept**, because they are different orders.
* `order_id` is stored as an integer.

In [20]:
silver_df = (step5_df
             .dropDuplicates()
             .withColumn("order_id", F.col("order_id").cast("int"))
             .orderBy("order_id", "customer_id"))

silver_df.printSchema()
silver_count = silver_df.count()
print("Rows in Silver:", silver_count)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)



Rows in Silver: 990


### Save to the Silver layer

In [21]:
# Spark writes a folder with part files. coalesce(1) gives a single part file
# for this small dataset. dateFormat keeps every date as yyyy-MM-dd.
(silver_df.coalesce(1)
 .write.mode("overwrite")
 .option("header", True)
 .option("dateFormat", "yyyy-MM-dd")
 .csv(SILVER_OUT))

# Also copy the part file to a single, easy-to-find CSV (the submission file).
part_file = [f for f in os.listdir(SILVER_OUT) if f.startswith("part-") and f.endswith(".csv")][0]
SILVER_CSV = os.path.join(SILVER_DIR, "sales_silver.csv")
shutil.copyfile(os.path.join(SILVER_OUT, part_file), SILVER_CSV)

print("Spark output folder:", sorted(os.listdir(SILVER_OUT)))
print("Single Silver CSV  :", SILVER_CSV)

Spark output folder: ['._SUCCESS.crc', '.part-00000-38d8045a-43fa-45ee-a8a1-2a508c604217-c000.csv.crc', '_SUCCESS', 'part-00000-38d8045a-43fa-45ee-a8a1-2a508c604217-c000.csv']
Single Silver CSV  : data_lake/silver/sales_silver.csv


## 5. Verification

In [22]:
print(f"Row count BEFORE preprocessing (Bronze): {raw_count}")
print(f"Row count AFTER  preprocessing (Silver): {silver_count}")
print(f"Rows removed (duplicates only)         : {raw_count - silver_count}")

Row count BEFORE preprocessing (Bronze): 1000
Row count AFTER  preprocessing (Silver): 990
Rows removed (duplicates only)         : 10


In [23]:
# Read the Silver data back with Spark
silver_read = (spark.read.option("header", True).option("inferSchema", True)
               .csv(SILVER_OUT))
silver_read.printSchema()
silver_read.show(10, truncate=False)
print("Rows read back from Silver:", silver_read.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+------------

In [24]:
# Re-run the discrepancy checks on the Silver data
check = silver_read.select(
    (F.lit(silver_read.count()) - F.lit(silver_read.distinct().count())).alias("exact_duplicates"),
    F.sum(F.when(F.col("customer_name").isNull() | F.col("payment_method").isNull() | F.col("city").isNull(), 1).otherwise(0)).alias("missing_text_values"),
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("quantity_<=_0"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("unit_price_<=_0"),
    F.sum(F.when((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100), 1).otherwise(0)).alias("discount_out_of_range"),
    F.sum(F.when(F.col("order_date").isNotNull() & ~F.col("order_date").cast("string").rlike(r"^\d{4}-\d{2}-\d{2}$"), 1).otherwise(0)).alias("non_standard_dates"),
)
check.show(truncate=False)

print("Distinct categories:", [r[0] for r in silver_read.select("category").distinct().orderBy("category").collect()])
print("Distinct cities    :", [r[0] for r in silver_read.select("city").distinct().orderBy("city").collect()])

+----------------+-------------------+-------------+---------------+---------------------+------------------+
|exact_duplicates|missing_text_values|quantity_<=_0|unit_price_<=_0|discount_out_of_range|non_standard_dates|
+----------------+-------------------+-------------+---------------+---------------------+------------------+
|0               |0                  |0            |0              |0                    |0                 |
+----------------+-------------------+-------------+---------------+---------------------+------------------+

Distinct categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Furniture', 'Grocery', 'Home Appliances', 'Sports']


Distinct cities    : ['Bengaluru', 'Chennai', 'Coimbatore', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Lucknow', 'Mumbai', 'Pune', 'Unknown', 'Vijayawada']


In [25]:
# Confirm the Bronze file has not been changed
assert md5(BRONZE_FILE) == BRONZE_MD5, "Bronze file was modified!"
print("Bronze file unchanged, md5:", md5(BRONZE_FILE))

print("\nData Lake structure:")
for root, dirs, files in sorted(os.walk(LAKE_DIR)):
    depth = root.count(os.sep)
    print("    " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        if not f.startswith("."):
            print("    " * (depth + 1) + f)

Bronze file unchanged, md5: 2b7d6737635600be42aa639140cd6ea9

Data Lake structure:
data_lake/
    bronze/
        sales.csv
    silver/
        sales_silver.csv
        preprocessed_sales/
            _SUCCESS
            part-00000-38d8045a-43fa-45ee-a8a1-2a508c604217-c000.csv


## 6. Summary of preprocessing performed

| # | Operation | How it was done |
|---|-----------|-----------------|
| 1 | **Removed exact duplicates** | `dropDuplicates()` on the raw rows, and again after text cleaning. Rows with a repeated `order_id` but different content were kept. |
| 2 | **Handled missing values** | Missing `customer_name` was filled from other rows with the same `customer_id`, otherwise set to `"Unknown"`. Missing `payment_method` and `city` were set to `"Unknown"`. No rows were dropped. |
| 3 | **Standardized text** | Leading, trailing and repeated spaces were removed from all text columns. `customer_name`, `category`, `city`, `state` and `order_status` were converted to Title Case (e.g. `ELECTRONICS` → `Electronics`, `' Hyderabad '` → `Hyderabad`). |
| 4 | **Validated numbers** | `quantity` must be a whole number > 0, `unit_price` > 0, and `discount_percent` between 0 and 100. Invalid values were set to NULL and the row was kept. |
| 5 | **Standardized dates** | `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy` and `MM-dd-yyyy` were parsed into one `DateType` column, saved as `yyyy-MM-dd`. Impossible dates (`not-a-date`, month 13, 31 Feb) were set to NULL. |
| 6 | **Kept useful records** | Only duplicate rows were removed. Unusual but possible values (for example, very high prices) were not changed. |

The Silver data is saved at `data_lake/silver/preprocessed_sales/` (Spark output) and `data_lake/silver/sales_silver.csv` (single CSV).

In [26]:
spark.stop()